# Project 1 — Gaussian & Laplacian Image Pyramids

Public portfolio version of coursework for the Image and Video Analysis course at NTUA. The original staff handout is not distributed here.

This notebook implements Gaussian and Laplacian image pyramids, reconstruction, entropy analysis and coefficient quantization, inspired by Burt & Adelson (1983).

## Imports

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import scipy.stats as st
from skimage import data, img_as_float


## Gaussian generating kernel

The parameter `a` controls the 1D generating kernel and therefore the amount of smoothing applied at each pyramid level.

In [ ]:
def GKernel(a):
    w = np.zeros(5)
    w[2] = a
    w[0] = 1/4 - a/2
    w[1] = 1/4
    w[4] = w[0]
    w[3] = w[1]
    return np.outer(w, w)


## Gaussian reduction and pyramid

In [ ]:
def GREDUCE(I, h):
    filtered = cv2.filter2D(I, -1, h)
    return cv2.resize(filtered, (filtered.shape[1]//2, filtered.shape[0]//2), interpolation=cv2.INTER_AREA)

def GPyramid(image, a, depth):
    h = GKernel(a)
    pyramid = [image]
    for _ in range(depth - 1):
        pyramid.append(GREDUCE(pyramid[-1], h))
    return pyramid


## Expansion and Laplacian pyramid

In [ ]:
def GEXPAND(I, h, target_shape=None):
    if target_shape is None:
        size = (I.shape[1]*2, I.shape[0]*2)
    else:
        size = (target_shape[1], target_shape[0])
    upsampled = cv2.resize(I, size, interpolation=cv2.INTER_LINEAR)
    return cv2.filter2D(upsampled, -1, 4*h)

def LPyramid(image, a, depth):
    gp = GPyramid(image, a, depth)
    lp = []
    h = GKernel(a)
    for i in range(depth - 1):
        expanded = GEXPAND(gp[i+1], h, gp[i].shape)
        lp.append(gp[i] - expanded)
    lp.append(gp[-1])
    return lp


## Reconstruction

Starting from the smallest Gaussian residual, each Laplacian level is expanded and added back to reconstruct the original image.

In [ ]:
def L_Pyramid_Decode(lp, a):
    h = GKernel(a)
    current = lp[-1]
    for level in reversed(lp[:-1]):
        current = GEXPAND(current, h, level.shape) + level
    return current


## Visualization helper

In [ ]:
def plot_pyramid(pyramid, cmap='gray'):
    fig, axes = plt.subplots(1, len(pyramid), figsize=(4*len(pyramid), 4))
    axes = np.atleast_1d(axes)
    for i, (ax, level) in enumerate(zip(axes, pyramid)):
        ax.imshow(level, cmap=None if level.ndim == 3 else cmap)
        ax.set_title(f'Level {i}')
        ax.axis('off')
    plt.tight_layout()


## Reconstruction experiment

The original work tested different values of `a` and pyramid depths on standard images such as Camera and Lena.

In [ ]:
camera = img_as_float(data.camera())
a = 0.4
depth = 5
lp = LPyramid(camera, a, depth)
reconstructed = L_Pyramid_Decode(lp, a)

print('Max absolute reconstruction error:', np.max(np.abs(camera - reconstructed)))
plot_pyramid(lp)


## Entropy analysis

For an 8-bit grayscale image, the theoretical maximum entropy is 8 bits/pixel when all 256 intensity values are equally likely. In the coursework, entropy was evaluated across different pyramid parameters to study compactness.

In [ ]:
def pyramid_entropy(image, a=0.4, depth=5, bins=256):
    lp = LPyramid(image, a, depth)
    values = np.concatenate([level.ravel() for level in lp[:-1]])
    hist, _ = np.histogram(values, bins=bins, density=False)
    p = hist / hist.sum()
    p = p[p > 0]
    return -(p * np.log2(p)).sum()

for a in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    print(a, pyramid_entropy(camera, a=a, depth=6))


## Quantization

A larger quantization bin reduces the number of representable coefficient values but increases information loss; a smaller bin preserves more detail at a higher coding cost.

In [ ]:
def L_Quantization(level, bin_size):
    return np.round(level / bin_size) * bin_size

def quantize_pyramid(lp, bin_size):
    return [L_Quantization(level, bin_size) for level in lp[:-1]] + [lp[-1]]

for bin_size in [8, 16, 32, 64]:
    qlp = quantize_pyramid(lp, bin_size)
    qrec = L_Pyramid_Decode(qlp, a)
    mse = np.mean((camera - qrec)**2)
    print(f'bin={bin_size:3d}  MSE={mse:.6f}')


## Main observations

The experiments showed successful reconstruction across the tested pyramid depths and kernel parameters before quantization. Entropy varied with both `a` and depth, while quantization introduced the expected trade-off between compactness and reconstruction fidelity.

**Reference:** P. J. Burt and E. H. Adelson, *The Laplacian Pyramid as a Compact Image Code*, IEEE Transactions on Communications, 1983.